# Phase Shifts

Phase shifts rotate the relative phase between the $|0\rangle$ and $|1\rangle$ components of a quantum state without changing measurement probabilities.

In [ ]:
import pennylane as qml
import numpy as np

## Define circuits

In [ ]:
dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev)
def with_phase(angle):
    qml.Hadamard(wires=0)
    qml.RZ(angle, wires=0)
    return qml.state()

@qml.qnode(dev)
def control_phase(a, b):
    qml.Hadamard(wires=0)
    qml.PhaseShift(a, wires=0)
    qml.Hadamard(wires=0)
    qml.PhaseShift(b, wires=0)
    return qml.state()

## Helper: extract relative phase

In [ ]:
def relative_phase(state_vec):
    """Extract the relative phase between |0> and |1> amplitudes."""
    amp_0, amp_1 = state_vec
    if abs(amp_0) < 1e-10:
        return 0.0
    return float(np.angle(amp_1 / amp_0))

## Phase shift sweep

Starting from $H|0\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$, apply $RZ(\theta)$ to rotate the relative phase.

In [ ]:
for angle in [0.0, np.pi / 4, np.pi / 2, np.pi, 3 * np.pi / 2]:
    sv = with_phase(angle)
    rp = relative_phase(sv)
    probs = np.abs(sv) ** 2
    print(f"RZ({angle:.4f}):  rel phase = {rp:.4f} rad ({np.degrees(rp):.1f} deg),  "
          f"P(|0>) = {probs[0]:.4f},  P(|1>) = {probs[1]:.4f}")

print()
print("Note: probabilities never change — only the phase does.")

## Phase accumulation

Two successive phase shifts add their phases.

In [ ]:
sv = control_phase(np.pi / 3, np.pi / 6)
rp = relative_phase(sv)
total = np.pi / 3 + np.pi / 6

print(f"PS(pi/3) then PS(pi/6):  rel phase = {rp:.4f} rad")
print(f"Expected total:          {total:.4f} rad")
print(f"Match: {np.isclose(rp, total)}")